# Explorer un jeu de données avec pandas

Première étape de toute analyse : **comprendre ce qu'on a entre les mains** avant d'écrire la moindre formule.  
On va charger un fichier CSV de consommation énergétique et répondre à ces questions :

1. À quoi ressemblent les données ?
2. Combien de lignes contient le fichier ?
3. Quel est le type de chaque colonne ?
4. Quelles sont les valeurs possibles pour les colonnes catégorielles ?

## Importer pandas

pandas est **la** bibliothèque Python pour manipuler des données tabulaires (lignes × colonnes).  
Par convention, on l'importe sous l'alias `pd`.

In [13]:
import pandas as pd

## Charger le fichier CSV

`pd.read_csv()` lit un fichier CSV et le transforme en **DataFrame** — le format central de pandas.  
Un DataFrame, c'est un tableau avec des colonnes nommées et un index numérique pour chaque ligne.

In [14]:
df = pd.read_csv("../data/consommation_energie.csv")

## Afficher les 5 premières lignes

`.head()` montre les premières lignes du DataFrame (5 par défaut).  
C'est le premier réflexe à avoir : vérifier que le fichier a été lu correctement.

In [15]:
df.head()

,date,batiment,zone,type_energie,kwh
0,2024-11-16,Usine_B,Production,gaz,347.87
1,2023-02-21,Usine_B,Production,gaz,785.20
2,2023-12-07,Entrepot_Nord,Production,electricite,406.46
3,2024-08-13,Entrepot_Nord,Eclairage,electricite,390.09
4,2023-11-28,Entrepot_Nord,Eclairage,electricite,605.13


## Compter le nombre total de lignes

`len(df)` renvoie le nombre de lignes du DataFrame.  
On peut aussi utiliser `df.shape` qui renvoie un tuple `(lignes, colonnes)`.

In [16]:
print(f"Nombre de lignes : {len(df)}")
print(f"Dimensions (lignes, colonnes) : {df.shape}")

Nombre de lignes : 2653
Dimensions (lignes, colonnes) : (2653, 5)


## Vérifier les types de chaque colonne

`.dtypes` indique le type de données de chaque colonne.  
C'est important : pandas ne devine pas toujours correctement. Par exemple, une colonne de dates sera souvent lue comme du texte (`object`) — il faudra la convertir explicitement.

In [17]:
df.dtypes

date                str
batiment            str
zone                str
type_energie        str
kwh             float64
dtype: object

## Lister les valeurs uniques des colonnes catégorielles

`.unique()` renvoie toutes les valeurs distinctes d'une colonne.  
`.nunique()` en donne le nombre.  
Cela permet de comprendre le périmètre des données : quels bâtiments ? quels types d'énergie ?

In [18]:
print(f"Bâtiments ({df['batiment'].nunique()} valeurs) :")
print(sorted(df["batiment"].unique()))

print(f"\nTypes d'énergie ({df['type_energie'].nunique()} valeurs) :")
print(sorted(df["type_energie"].unique()))

Bâtiments (4 valeurs) :
['Bureau_Central', 'Entrepot_Nord', 'Usine_A', 'Usine_B']

Types d'énergie (2 valeurs) :
['electricite', 'gaz']


## Résumé

| Ce qu'on a appris | Détail |
|---|---|
| **Lignes** | 2 653 enregistrements |
| **Colonnes** | `date`, `batiment`, `zone`, `type_energie`, `kwh` |
| **Types** | 4 colonnes texte + 1 colonne numérique |
| **Bâtiments** | 4 (Bureau_Central, Entrepot_Nord, Usine_A, Usine_B) |
| **Énergies** | 2 (electricite, gaz) |

**Prochaine étape** : convertir la colonne `date` et commencer à visualiser les données.

---

### Mini-exercice

Afficher les valeurs uniques de la colonne `zone` et compter combien il y en a.

In [19]:
# À compléter :
# print(df["zone"]...)

---

## Visualiser : consommation totale par bâtiment

Maintenant qu'on connaît nos données, on passe à la **visualisation**.
L'objectif : comparer la consommation totale en kWh de chaque bâtiment avec un graphique en barres.

On va procéder en 2 étapes :
1. Calculer la consommation totale par bâtiment avec `.groupby()`
2. Créer le graphique en barres trié

### Étape 1 — Calculer la consommation totale par bâtiment

`.groupby("batiment")` regroupe les lignes par bâtiment, puis `["kwh"].sum()` additionne les kWh de chaque groupe.

**Concept fondamental** : `groupby` est l'équivalent pandas du `GROUP BY` en SQL. C'est l'outil central pour agréger des données par catégorie (somme, moyenne, comptage…).

On trie ensuite par valeur décroissante avec `.sort_values(ascending=False)` — le bâtiment le plus gourmand en premier.

In [21]:
# Regrouper par bâtiment et sommer les kWh
conso_par_batiment = (
    df.groupby("batiment")["kwh"]
    .sum()
    .sort_values(ascending=False)
)

conso_par_batiment

batiment
Entrepot_Nord     315668.528920
Usine_A           308827.886303
Usine_B           282608.720505
Bureau_Central    270367.801651
Name: kwh, dtype: float64

### Étape 2 — Créer le graphique en barres

On utilise `plotly.express` pour créer le graphique rapidement, puis on ajuste les détails :
- **Titre actif** : il décrit ce qu'on observe, pas juste "Consommation par bâtiment"
- **Labels sur les barres** : `text_auto=True` affiche la valeur directement → pas besoin de lire l'axe Y
- **Barres triées** : le tri est déjà fait dans les données, Plotly respecte l'ordre

In [22]:
import plotly.express as px

fig = px.bar(
    x=conso_par_batiment.index,
    y=conso_par_batiment.values,
    text_auto=".0f",  # Affiche les valeurs arrondies sur chaque barre
)

fig.update_traces(
    textposition="outside",  # Labels au-dessus des barres
)

fig.update_layout(
    title="Consommation totale par bâtiment (kWh)",
    xaxis_title="Bâtiment",
    yaxis_title="Consommation totale (kWh)",
    width=800,
    height=500,
    showlegend=False,
)

# Afficher dans le notebook ET sauvegarder en HTML
fig.show()
fig.write_html("../output/conso_par_batiment.html")
print("Graphique sauvegardé dans output/conso_par_batiment.html")

---

## Visualiser : évolution mensuelle de la consommation par bâtiment

Le graphique en barres montre le **total** — mais pas comment la consommation évolue dans le temps.
Un **graphique en lignes** est le bon choix pour visualiser une évolution temporelle.

On va :
1. Convertir la colonne `date` en type datetime (pandas la lit comme du texte par défaut)
2. Créer une colonne `mois` pour regrouper par période mensuelle
3. Agréger les kWh par mois et par bâtiment
4. Tracer une ligne par bâtiment avec des marqueurs sur chaque point

**Concept fondamental** : `pd.to_datetime()` transforme du texte en dates exploitables. Sans cette conversion, pandas ne sait pas trier chronologiquement ni regrouper par mois.

### Étape 1 — Préparer les données mensuelles

Trois opérations enchaînées :
- `pd.to_datetime()` : convertit le texte `"2024-01-15"` en objet date manipulable
- `.dt.to_period("M")` : extrait la période mensuelle (ex : `2024-01`) — plus adapté que `.dt.month` car on garde l'année
- `.groupby()` sur deux colonnes : on agrège par **mois ET bâtiment** simultanément

**Concept fondamental** : `.pivot_table()` réorganise un DataFrame "long" (une ligne par observation) en tableau "large" (une colonne par catégorie). C'est l'équivalent d'un tableau croisé dynamique dans Excel.

In [23]:
# Convertir la colonne date en datetime
df["date"] = pd.to_datetime(df["date"])

# Créer une colonne "mois" au format période mensuelle (ex : 2024-01)
df["mois"] = df["date"].dt.to_period("M")

# Agréger : somme des kWh par mois et par bâtiment
# pivot_table crée un tableau avec les mois en lignes et les bâtiments en colonnes
conso_mensuelle = df.pivot_table(
    values="kwh",
    index="mois",
    columns="batiment",
    aggfunc="sum",
)

# Trier par ordre chronologique (normalement déjà trié, mais on s'assure)
conso_mensuelle = conso_mensuelle.sort_index()

# Aperçu des premières lignes
conso_mensuelle.head()

batiment,Bureau_Central,Entrepot_Nord,Usine_A,Usine_B
mois,,,,
2023-01,10278.409934,11833.42,10312.89,11657.92
2023-02,12015.760000,12664.86,12198.83,10693.33
2023-03,13029.620000,12367.58,14817.49,13266.78
2023-04,7862.080000,12982.63,12503.54,13377.33
2023-05,8791.180000,10608.69,12208.95,9900.07


### Étape 2 — Tracer l'évolution mensuelle avec un graphique en lignes

On utilise `go.Scatter` (mode `lines+markers`) pour chaque bâtiment. Pourquoi pas `px.line` ? Parce qu'on veut un contrôle fin sur les marqueurs et les couleurs de chaque ligne.

Points de style appliqués :
- **Marqueurs** sur chaque point → repérer facilement les mois exacts
- **4 bâtiments = 4 lignes** → dans la limite des 4-5 séries recommandées
- **Axe X** : les mois en ordre chronologique
- **`hovermode="x unified"`** : au survol d'un mois, on voit tous les bâtiments d'un coup

In [24]:
import plotly.express as px

# Convertir l'index Period en chaînes pour l'axe X
mois_labels = conso_mensuelle.index.astype(str)

# Transformer en format long pour px.line (une colonne par variable)
conso_long = conso_mensuelle.copy()
conso_long.index = mois_labels
conso_long = conso_long.reset_index().melt(
    id_vars="mois", var_name="batiment", value_name="kwh"
)

# Noms plus lisibles
conso_long["batiment"] = conso_long["batiment"].str.replace("_", " ")

fig = px.line(
    conso_long,
    x="mois",
    y="kwh",
    color="batiment",
    markers=True,  # Marqueurs sur chaque point
    title="Évolution mensuelle de la consommation par bâtiment",
    labels={"kwh": "Consommation (kWh)", "mois": "Mois", "batiment": "Bâtiment"},
)

fig.update_layout(width=900, height=550)

fig.show()
fig.write_html("../output/evolution_mensuelle_kwh.html")
print("Graphique sauvegardé dans output/evolution_mensuelle_kwh.html")

---

### Mini-exercice

Modifier le code ci-dessus pour afficher uniquement l'évolution de l'**électricité** (filtrer `df[df["type_energie"] == "electricite"]` avant le `pivot_table`). Observer si les tendances changent par rapport au graphique toutes énergies confondues.

---

## Visualiser : répartition hiérarchique avec un Sunburst

Le graphique **sunburst** (ou "rayon de soleil") affiche des données hiérarchiques sous forme d'anneaux concentriques.
Chaque anneau représente un niveau de la hiérarchie : plus on s'éloigne du centre, plus on descend dans le détail.

Ici, on va visualiser **3 niveaux** :
1. **Centre** → Bâtiment (qui consomme le plus ?)
2. **Anneau intermédiaire** → Zone (Production, Climatisation, Éclairage, Bureaux)
3. **Anneau extérieur** → Type d'énergie (électricité vs gaz)

**Quand utiliser un sunburst ?**
- Pour montrer une **décomposition progressive** : du global vers le détail
- Quand la hiérarchie a **2 à 3 niveaux** (au-delà, ça devient illisible)
- Pour repérer visuellement les **proportions relatives** à chaque niveau

**Concept fondamental** : `px.sunburst()` attend un DataFrame avec une colonne par niveau hiérarchique et une colonne de valeurs. Le paramètre `path=` définit l'ordre des niveaux (du centre vers l'extérieur).

In [25]:
# Agréger les kWh par bâtiment, zone et type d'énergie
conso_hierarchie = (
    df.groupby(["batiment", "zone", "type_energie"])["kwh"]
    .sum()
    .reset_index()
)

# Renommer pour un affichage plus lisible
conso_hierarchie["batiment"] = conso_hierarchie["batiment"].str.replace("_", " ")
conso_hierarchie["type_energie"] = conso_hierarchie["type_energie"].str.capitalize()

# Créer le sunburst : path= définit les niveaux du centre vers l'extérieur
fig = px.sunburst(
    conso_hierarchie,
    path=["batiment", "zone", "type_energie"],
    values="kwh",
)

fig.update_traces(
    textinfo="label+percent parent",
    insidetextorientation="radial",
)

fig.update_layout(
    title="Répartition de la consommation : bâtiment > zone > énergie",
    width=800,
    height=700,
    margin=dict(l=20, r=20, t=80, b=20),
)

fig.show()
fig.write_html("../output/sunburst_consommation.html")
print("Graphique sauvegardé dans output/sunburst_consommation.html")

---

### Mini-exercice

Modifier le sunburst pour n'afficher que **2 niveaux** : bâtiment > zone (sans le type d'énergie).
Indice : il suffit de changer le paramètre `path=` dans `px.sunburst()`.

---

## Nouveau jeu de données : production industrielle

On change de fichier. `production_industrielle.csv` contient des données de production quotidienne :
lignes de production, quantités produites, défauts détectés, temps d'arrêt et équipes.

**Objectif** : explorer la relation entre la quantité produite et le nombre de défauts.
Est-ce que produire plus signifie plus de défauts ? La réponse dépend-elle de l'équipe ?

On va utiliser un **nuage de points (scatter plot)** — le bon choix quand on veut voir
la **corrélation** entre deux variables numériques.

In [26]:
# Charger le fichier de production industrielle
df_prod = pd.read_csv("../data/production_industrielle.csv")

# Premier réflexe : regarder les données
print(f"Dimensions : {df_prod.shape[0]} lignes × {df_prod.shape[1]} colonnes")
df_prod.head()

Dimensions : 1492 lignes × 7 colonnes


,date,ligne,produit,quantite_produite,quantite_defauts,temps_arret_min,operateur
0,2024-01-01,Ligne_1,Produit_A,277,13,3.0,Equipe_Matin
1,2024-01-01,Ligne_1,Produit_B,474,25,10.0,Equipe_Nuit
2,2024-01-01,Ligne_2,Produit_A,282,8,48.0,Equipe_Apres_Midi
3,2024-01-01,Ligne_2,Produit_B,200,7,17.0,Equipe_Matin
4,2024-01-01,Ligne_2,Produit_C,144,12,10.0,Equipe_Nuit


### Créer le nuage de points coloré par équipe

`px.scatter()` place un point par ligne du DataFrame :
- **x** → quantité produite (variable explicative)
- **y** → nombre de défauts (variable qu'on cherche à comprendre)
- **color** → équipe (chaque équipe aura sa propre couleur)
- **trendline="ols"** → ajoute une droite de régression linéaire (OLS = Ordinary Least Squares)

**Concept fondamental** : la **régression linéaire** trace la droite qui minimise
la distance entre elle et tous les points. Si la droite monte, il y a une corrélation positive
(plus on produit, plus il y a de défauts). Si elle est plate, pas de lien apparent.

In [27]:
import plotly.express as px

# Renommer les équipes pour un affichage plus lisible
df_prod["equipe"] = (
    df_prod["operateur"]
    .str.replace("Equipe_", "")        # Retirer le préfixe
    .str.replace("_", " ")             # Remplacer les underscores
)

# Créer le nuage de points avec ligne de tendance par équipe
fig = px.scatter(
    df_prod,
    x="quantite_produite",
    y="quantite_defauts",
    color="equipe",                     # Une couleur par équipe
    trendline="ols",                    # Droite de régression par groupe
    opacity=0.6,                        # Transparence pour voir les superpositions
    title="Plus on produit, plus on détecte de défauts — quelle que soit l'équipe",
    labels={
        "quantite_produite": "Quantité produite",
        "quantite_defauts": "Nombre de défauts",
        "equipe": "Équipe",
    },
)

fig.update_layout(
    width=800,
    height=500,
)

# Afficher dans le notebook ET sauvegarder en HTML
fig.show()
fig.write_html("../output/scatter_production_defauts.html")
print("Graphique sauvegardé dans output/scatter_production_defauts.html")

### Comment lire ce graphique ?

- **Chaque point** = un jour de production sur une ligne donnée
- **Position horizontale** = combien d'unités produites ce jour-là
- **Position verticale** = combien de défauts détectés
- **Couleur** = quelle équipe était en poste
- **Droites** = tendance linéaire pour chaque équipe

Si les droites montent de gauche à droite, cela confirme une **corrélation positive** :
les jours de forte production génèrent davantage de défauts.
Si une équipe a une droite plus pentue que les autres, elle a un taux de défauts
qui augmente plus vite avec le volume.

---

### Mini-exercice

Remplacer `color="equipe"` par `color="produit"` (colonne `produit` du DataFrame)
pour voir si certains produits génèrent plus de défauts que d'autres à volume égal.